# Industrial Pump Condition Classification

**AI Services MS2 | Supplementary Classification Practice | 60 marks | 20 tests**

A maintenance team must classify pump inspections as **Healthy**, **Inspect**, or **Urgent**. Complete the independent functions, compare two classifiers on held-back data, save the winning fitted pipeline and score queued inspections without retraining.

**Input files:** `pump_train.csv`, `pump_test.csv`, `pump_predict.csv`. Keep all three beside this notebook. The model/report outputs are generated only when the workflow cells are run.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

TARGET = "condition_label"
CLASS_NAMES = ["Healthy", "Inspect", "Urgent"]
raw_numeric_cols = [
    "inlet_pressure_bar", "outlet_pressure_bar", "flow_volume_m3",
    "runtime_hours", "motor_temperature_c", "ambient_temperature_c",
    "vibration_mm_s", "operating_load_pct"
]
category_cols = ["pump_type", "maintenance_zone", "shift_code"]
numeric_features = raw_numeric_cols + [
    "inspection_hour", "pressure_gain_bar", "flow_rate_m3h", "temperature_rise_c"
]
features = numeric_features + category_cols

## Function A1 - Common loader

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
#     """Clean one CSV and create pump features. See instruction paper."""
#     # TODO: IDs, timestamps, numeric/category cleaning and 4 engineered features
#     # raise NotImplementedError
    
    df=pd.read_csv(filename,dtype={"inspection_id":str})

    df["inspection_id"]=df["inspection_id"].str.strip().replace("",np.nan)
    df=df.dropna(subset=["inspection_id"])
    df=df.drop_duplicates(subset=["inspection_id"],keep="first").copy()

    df["inspected_at"]=pd.to_datetime(df["inspected_at"],format="mixed",dayfirst=True,errors="coerce")

    df = df.dropna(subset="inspected_at")
    df["inspection_hour"]=df["inspected_at"].dt.hour

    for col in raw_numeric_cols:
        df[col]=pd.to_numeric(df[col],errors="coerce").replace([np.inf,-np.inf],np.nan)


    for col in category_cols:
        df[col]=df[col].astype("string").str.strip().str.lower().replace("", pd.NA).fillna("unknown")
        
    df["shift_code"]=df["shift_code"].replace(r"\.0+$","",regex=True).astype("string")


    df["pressure_gain_bar"]=df["outlet_pressure_bar"]-df["inlet_pressure_bar"]
    df["temperature_rise_c"]=df["motor_temperature_c"]-df["ambient_temperature_c"]

    runtime = df["runtime_hours"].where(df["runtime_hours"] > 0)

    df["flow_rate_m3h"]=df["flow_volume_m3"]/runtime

    # df["flow_rate_m3h"]=df["flow_rate_m3h"].replace([np.inf,-np.inf],np.nan)
    
    return df
    

    
    

## Function A2 - Reusable classification pipeline

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Return unfitted preprocessing plus the passed classifier."""
    # TODO: ColumnTransformer with numeric and categorical branches
    num_col=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ])

    cat_col=ColumnTransformer([
        ("numeric",num_col,numeric_features),
        ("category",OneHotEncoder(handle_unknown="ignore"),category_cols)
    ])

    return Pipeline([("preprocessor",cat_col),("model",algorithm)])

    

    

## Function 1 - Prepare labeled datasets

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def prepare_datasets(
    train_file: str, test_file: str
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Load files, normalize valid classes and return X_train, X_test, y_train, y_test."""
    # # TODO: Filter targets and use one stable LabelEncoder mapping
    # raise NotImplementedError
    train_df=load_data(train_file)
    test_df=load_data(test_file)
    encoder=LabelEncoder().fit(CLASS_NAMES)

    train_df[TARGET]=train_df[TARGET].str.strip().str.title().astype("string")
    test_df[TARGET]=test_df[TARGET].str.strip().str.title().astype("string")

    train_df=train_df[train_df[TARGET].isin(CLASS_NAMES)]
    test_df=test_df[test_df[TARGET].isin(CLASS_NAMES)]

    y_train= pd.Series(encoder.transform(train_df[TARGET]),index=train_df.index,name=TARGET)
    y_test= pd.Series(encoder.transform(test_df[TARGET]),index=test_df.index,name=TARGET)
    
    return train_df[features],test_df[features],y_train,y_test
    

## Function 2 - Train candidate classifiers

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def train_models(
    X_train: pd.DataFrame, y_train: pd.Series
) -> dict[str, Pipeline]:
    """Fit Logistic Regression and Random Forest classifier pipelines."""
    # # TODO: Compare the two prescribed classifiers
    # raise NotImplementedError

    models={
        "logistic_regression":LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42),
        "random_forest":RandomForestClassifier(n_estimators=150,max_depth=12,min_samples_leaf=2,class_weight="balanced",random_state=42,n_jobs=-1)
    }

    
    trained={}
    for name,algo in models.items():
        trained[name]=build_pipeline(algo).fit(X_train,y_train)

    return trained
        

## Function 3 - Evaluate and compare models

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def evaluate_models(
    trained_models: dict[str, Pipeline], X_test: pd.DataFrame,
    y_test: pd.Series
) -> pd.DataFrame:
    """Return accuracy, macro F1 and weighted F1, sorted by macro F1."""
    # # TODO: Rank without refitting
    # raise NotImplementedError
    
    result=[]
    for name,model in trained_models.items():
        pred=model.predict(X_test)
        result.append({
            "model_name":name,
            "accuracy":accuracy_score(y_test,pred),
            "macro_f1":f1_score(y_test,pred,average="macro",labels=[0,1,2],zero_division=0),
            "weighted_f1":f1_score(y_test,pred,average="weighted",labels=[0,1,2],zero_division=0)
        })
    comp=pd.DataFrame(result)

    return comp.sort_values(["macro_f1","accuracy","model_name"],ascending=[False,False,True]).reset_index(drop=True)

    



## Function 4 - Select best model

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def select_best_model(
    comparison: pd.DataFrame, trained_models: dict[str, Pipeline]
) -> tuple[str, Pipeline]:
    """Return first-ranked model name and the same fitted pipeline."""
    best_name=comparison.iloc[0]["model_name"]
    best_model=trained_models[best_name]

    return best_name,best_model

## Function 5 - Predict pump condition

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def predict_condition(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return labels and Urgent-class probabilities for cleaned queued records."""
    # TODO: inverse_transform and predict_proba
    raise NotImplementedError

## Training workflow

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Persist model, comparison and confusion matrix when called."""
    raise NotImplementedError

## Prediction workflow

See `../instructions/instructions.pdf` for the detailed behavior, exact inputs/outputs and grading checks.

In [ ]:
def predict() -> pd.DataFrame:
    """Reload saved pipeline and export queued predictions when called."""
    raise NotImplementedError

## Run the workflows

Run the **training** cell first. You may run **prediction** later in a fresh kernel after executing only the imports and definition cells, provided `best_model.pkl` is present. Do not fit inside the prediction workflow.

In [ ]:
train()

In [ ]:
predict()